In [1]:
# journal_08_qualitative.ipynb -- one cell. Remakes the qualitative figures with visually different scenes.
# Runs A, B s1 and Thermal only over the validation images (a few minutes on the GPU), then picks
# 3 "fusion helps" images and 1 "fusion hurts" image that are NOT near-duplicate frames of each other
# (FLIR/M3FD come from video, so consecutive frames look alike), skipping crowded images (> MAX_GT objects)
# whose boxes are unreadable at print size.
# Writes journal\figures\qualitative_flir_v2.png and qualitative_m3fd_v2.png. Nothing else is changed;
# the old figures and analysis_*.json are left as they are.
import cv2
import numpy as np
import torch

import lb_fusion as L

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_GT = 20                         # skip images with more ground-truth objects than this
SIMILARITY_STEPS = [0.5, 0.6, 0.7, 0.8, 0.9]   # loosened only if not enough distinct scenes are found
KEEP = ["A (RGB only = detached adapter)", "B s1", "Thermal only"]


def thumbnail(ds, img_id):
    g = cv2.imread(str(ds["root"] / "images" / "val" / f"{img_id}{ds['ext']}"), cv2.IMREAD_GRAYSCALE)
    t = cv2.resize(g, (32, 24), interpolation=cv2.INTER_AREA).astype(np.float32).ravel()
    return (t - t.mean()) / (t.std() + 1e-6)


def pick_rows(ds, preds, gts, pool, n_help=3, conf=0.25):
    a, b = preds[KEEP[0]], preds[KEEP[1]]
    diff = {i: L.per_image_hits(b[i], gts[i], conf) - L.per_image_hits(a[i], gts[i], conf) for i in pool}
    cand = [i for i in pool if 0 < len(gts[i][0]) <= MAX_GT]
    thumbs = {}
    for limit in SIMILARITY_STEPS:
        chosen = []

        def distinct(i):
            if i not in thumbs:
                thumbs[i] = thumbnail(ds, i)
            return all(float(thumbs[i] @ thumbs[j]) / len(thumbs[i]) < limit for j in chosen)

        for i in sorted(cand, key=lambda i: -diff[i]):
            if len(chosen) == n_help or diff[i] <= 0:
                break
            if distinct(i):
                chosen.append(i)
        for i in sorted(cand, key=lambda i: diff[i]):
            if diff[i] >= 0:
                break
            if distinct(i):
                chosen.append(i)
                break
        if len(chosen) == n_help + 1:
            print(f"  picked with scene-similarity limit {limit}")
            break
    for i in chosen:
        print(f"  {i}: B finds {diff[i]:+d} objects vs A ({len(gts[i][0])} ground-truth objects)")
    return chosen


for name in ["flir", "m3fd"]:
    ds = L.DATASETS[name]
    _, _, eval_sets = L.get_splits(name, smoke=False)
    ids = eval_sets["full"]
    pool = eval_sets.get("night", ids)            # FLIR: night images only, as before; M3FD: all
    specs = {k: v for k, v in L._analysis_specs(name).items() if k in KEEP}
    preds, gts = {}, None
    for model_name, spec in specs.items():
        print(f"\n=== {name} | {model_name} ===")
        model = L._load_eval_model(spec, name, DEVICE)
        p, g, _, _ = L.collect_predictions(model, ds, pool, DEVICE)
        preds[model_name], gts = p, (gts or g)
        del model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
    print(f"\nchoosing images for {name}:")
    rows = pick_rows(ds, preds, gts, pool)
    if len(rows) < 4:
        print(f"only {len(rows)} suitable images found for {name} -- send me this output")
        continue
    # make_qualitative_figure puts the 3 largest gains first and the largest loss last, so handing it
    # exactly these 4 images reproduces the chosen rows
    L.make_qualitative_figure(ds, preds, gts, rows, L.JOURNAL / "figures" / f"qualitative_{name}_v2.png")

loaded holdout split: C:\Users\LENOVO\Documents\Thermal_Fusion_Paper\journal\splits\flir_train_holdout.json

=== flir | A (RGB only = detached adapter) ===
  batch 0/40...
  batch 20/40...

=== flir | B s1 ===
  batch 0/40...
  batch 20/40...

=== flir | Thermal only ===
  batch 0/40...
  batch 20/40...

choosing images for flir:
  picked with scene-similarity limit 0.5
  FLIR_08885: B finds +7 objects vs A (15 ground-truth objects)
  FLIR_09403: B finds +7 objects vs A (11 ground-truth objects)
  FLIR_08934: B finds +5 objects vs A (14 ground-truth objects)
  FLIR_09390: B finds -1 objects vs A (3 ground-truth objects)
figure saved: C:\Users\LENOVO\Documents\Thermal_Fusion_Paper\journal\figures\qualitative_flir_v2.png
loaded holdout split: C:\Users\LENOVO\Documents\Thermal_Fusion_Paper\journal\splits\m3fd_train_holdout.json
(skipping B_robust s2: C:\Users\LENOVO\Documents\Thermal_Fusion_Paper\checkpoints_lb\lb_yolo_m3fd_B_robust_s2_best.pt not found)

=== m3fd | A (RGB only = detached